In [ ]:
# ============================================================
# GOOGLE COLAB - SINGLE CELL
#
# LANGCHAIN + LANGGRAPH + GROQ + GRADIO
# DYNAMIC RAG + DYNAMIC DOCUMENT GRAPH
#
# LLM:
#     openai/gpt-oss-120b
#
# Features:
#   1. Groq API key entered in UI
#   2. Upload PDF / DOCX / TXT / MD / CSV
#   3. LangChain document loading
#   4. Chunking
#   5. HuggingFace embeddings
#   6. FAISS vector store
#   7. LangGraph RAG workflow
#   8. Dynamic document knowledge graph
#   9. Chatbot using Gradio messages format
#  10. Graph updates after every upload/build
# ============================================================


# ============================================================
# 1. INSTALL ALL PACKAGES
# ============================================================

!pip -q install -U \
    langchain \
    langchain-core \
    langchain-community \
    langchain-groq \
    langchain-huggingface \
    langchain-text-splitters \
    langgraph \
    faiss-cpu \
    sentence-transformers \
    pypdf \
    python-docx \
    pandas \
    matplotlib \
    networkx \
    gradio


# ============================================================
# 2. IMPORTS
# ============================================================

import os
import traceback
from pathlib import Path
from typing import TypedDict, Any

import gradio as gr
import matplotlib.pyplot as plt
import networkx as nx

from langchain_groq import ChatGroq

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    CSVLoader,
    Docx2txtLoader
)

from langchain_community.vectorstores import FAISS

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import (
    StateGraph,
    START,
    END
)


# ============================================================
# 3. GLOBAL APPLICATION STATE
# ============================================================

VECTORSTORE = None

ALL_CHUNKS = []

UPLOADED_FILES = []

DOCUMENT_COUNT = 0

CHUNK_COUNT = 0

GRAPH = None


# ============================================================
# 4. EMBEDDING MODEL
# ============================================================

print("Loading embedding model...")

embeddings = HuggingFaceEmbeddings(

    model_name=
    "sentence-transformers/all-MiniLM-L6-v2",

    model_kwargs={
        "device": "cpu"
    },

    encode_kwargs={
        "normalize_embeddings": True
    }
)

print("Embedding model loaded.")


# ============================================================
# 5. LOAD SINGLE FILE
# ============================================================

def load_single_file(file_path):

    suffix = Path(
        file_path
    ).suffix.lower()

    if suffix == ".pdf":

        loader = PyPDFLoader(
            file_path
        )

    elif suffix in [".txt", ".md"]:

        loader = TextLoader(
            file_path,
            encoding="utf-8",
            autodetect_encoding=True
        )

    elif suffix == ".csv":

        loader = CSVLoader(
            file_path
        )

    elif suffix == ".docx":

        loader = Docx2txtLoader(
            file_path
        )

    else:

        raise ValueError(
            f"Unsupported file format: {suffix}"
        )

    documents = loader.load()

    for doc in documents:

        doc.metadata[
            "source_file"
        ] = Path(
            file_path
        ).name

    return documents


# ============================================================
# 6. LOAD ALL FILES
# ============================================================

def load_all_files(files):

    if not files:

        raise ValueError(
            "Please upload at least one file."
        )

    documents = []

    for file_path in files:

        docs = load_single_file(
            file_path
        )

        documents.extend(
            docs
        )

    return documents


# ============================================================
# 7. BUILD KNOWLEDGE BASE
# ============================================================

def build_knowledge_base(files):

    global VECTORSTORE
    global ALL_CHUNKS
    global UPLOADED_FILES
    global DOCUMENT_COUNT
    global CHUNK_COUNT

    try:

        # ----------------------------------------
        # Reset
        # ----------------------------------------

        VECTORSTORE = None

        ALL_CHUNKS = []

        UPLOADED_FILES = []

        # ----------------------------------------
        # Load
        # ----------------------------------------

        documents = load_all_files(
            files
        )

        DOCUMENT_COUNT = len(
            documents
        )

        UPLOADED_FILES = [
            Path(f).name
            for f in files
        ]

        # ----------------------------------------
        # Split
        # ----------------------------------------

        splitter = RecursiveCharacterTextSplitter(

            chunk_size=800,

            chunk_overlap=100,

            separators=[
                "\n\n",
                "\n",
                ". ",
                " ",
                ""
            ]
        )

        chunks = splitter.split_documents(
            documents
        )

        if not chunks:

            raise ValueError(
                "No text could be extracted."
            )

        # ----------------------------------------
        # Add chunk IDs
        # ----------------------------------------

        for index, chunk in enumerate(
            chunks
        ):

            chunk.metadata[
                "chunk_id"
            ] = index + 1

        ALL_CHUNKS = chunks

        CHUNK_COUNT = len(
            chunks
        )

        # ----------------------------------------
        # FAISS
        # ----------------------------------------

        VECTORSTORE = FAISS.from_documents(

            chunks,

            embeddings
        )

        # ----------------------------------------
        # Dynamic graph
        # ----------------------------------------

        graph_image = create_document_graph(
            documents,
            chunks
        )

        # ----------------------------------------
        # Status
        # ----------------------------------------

        status = (
            "## ✅ Knowledge Base Created\n\n"
            f"**Files:** {len(files)}  \n"
            f"**Documents/Pages:** {DOCUMENT_COUNT}  \n"
            f"**Chunks:** {CHUNK_COUNT}\n\n"
            "### Uploaded Files\n\n"
        )

        for filename in UPLOADED_FILES:

            status += (
                f"- 📄 `{filename}`\n"
            )

        status += (
            "\n### RAG Configuration\n\n"
            "- Embeddings: `all-MiniLM-L6-v2`\n"
            "- Vector DB: `FAISS`\n"
            "- Retrieval: top 4 chunks\n"
            "- LLM: `openai/gpt-oss-120b`"
        )

        return (
            status,
            graph_image
        )

    except Exception as e:

        print(
            traceback.format_exc()
        )

        VECTORSTORE = None

        return (
            f"## ❌ Build Error\n\n`{str(e)}`",
            None
        )


# ============================================================
# 8. CREATE DYNAMIC DOCUMENT GRAPH
# ============================================================
#
# This graph is generated from the ACTUAL uploaded data.
#
# Structure:
#
# File
#   |
#   +--> Chunk 1
#   +--> Chunk 2
#   +--> Chunk 3
#
# For multiple files:
#
# File A ---> chunks
# File B ---> chunks
# File C ---> chunks
#
# This means every upload produces a NEW graph.
# ============================================================

def create_document_graph(
    documents,
    chunks
):

    try:

        G = nx.DiGraph()

        # ----------------------------------------
        # Add source nodes
        # ----------------------------------------

        source_files = []

        for doc in documents:

            source = doc.metadata.get(
                "source_file",
                "Unknown"
            )

            if source not in source_files:

                source_files.append(
                    source
                )

        # ----------------------------------------
        # File nodes
        # ----------------------------------------

        for source in source_files:

            G.add_node(

                f"FILE::{source}",

                node_type="file",

                label=source
            )

        # ----------------------------------------
        # Chunk nodes
        # ----------------------------------------

        for chunk in chunks:

            chunk_id = chunk.metadata.get(
                "chunk_id"
            )

            source = chunk.metadata.get(
                "source_file",
                "Unknown"
            )

            chunk_node = (
                f"CHUNK::{chunk_id}"
            )

            preview = (
                chunk.page_content
                .replace("\n", " ")
                .strip()
            )

            preview = preview[:100]

            G.add_node(

                chunk_node,

                node_type="chunk",

                label=(
                    f"Chunk {chunk_id}\n"
                    f"{preview}"
                )
            )

            # File -> Chunk

            G.add_edge(

                f"FILE::{source}",

                chunk_node
            )

        # ----------------------------------------
        # Add RAG nodes
        # ----------------------------------------

        G.add_node(

            "VECTOR",

            node_type="vector",

            label="FAISS\nVector Store"
        )

        G.add_node(

            "RETRIEVER",

            node_type="retriever",

            label="Retriever\nTop-K"
        )

        G.add_node(

            "LLM",

            node_type="llm",

            label="Groq\nGPT-OSS-120B"
        )

        # ----------------------------------------
        # Connect chunks to vector store
        # ----------------------------------------

        for chunk in chunks:

            chunk_id = chunk.metadata.get(
                "chunk_id"
            )

            G.add_edge(

                f"CHUNK::{chunk_id}",

                "VECTOR"
            )

        # ----------------------------------------
        # RAG flow
        # ----------------------------------------

        G.add_edge(
            "VECTOR",
            "RETRIEVER"
        )

        G.add_edge(
            "RETRIEVER",
            "LLM"
        )

        # ----------------------------------------
        # Draw
        # ----------------------------------------

        fig, ax = plt.subplots(
            figsize=(16, 10)
        )

        # Layered positions
        pos = {}

        # ----------------------------------------
        # File positions
        # ----------------------------------------

        file_nodes = [
            n for n, d in G.nodes(
                data=True
            )
            if d["node_type"] == "file"
        ]

        for i, node in enumerate(
            file_nodes
        ):

            pos[node] = (
                0,
                -i * 2
            )

        # ----------------------------------------
        # Chunk positions
        # ----------------------------------------

        chunk_nodes = [
            n for n, d in G.nodes(
                data=True
            )
            if d["node_type"] == "chunk"
        ]

        chunks_by_file = {}

        for node in chunk_nodes:

            source = None

            for parent in G.predecessors(
                node
            ):

                if G.nodes[parent][
                    "node_type"
                ] == "file":

                    source = parent

                    break

            if source is not None:

                chunks_by_file.setdefault(
                    source,
                    []
                ).append(node)

        for file_index, (
            source,
            node_list
        ) in enumerate(
            chunks_by_file.items()
        ):

            for chunk_index, node in enumerate(
                node_list
            ):

                pos[node] = (
                    3,
                    -(file_index * 3)
                    -
                    chunk_index * 0.7
                )

        # ----------------------------------------
        # RAG nodes
        # ----------------------------------------

        pos["VECTOR"] = (
            6,
            -2
        )

        pos["RETRIEVER"] = (
            8.5,
            -2
        )

        pos["LLM"] = (
            11,
            -2
        )

        # ----------------------------------------
        # Node grouping
        # ----------------------------------------

        file_group = [
            n for n, d in G.nodes(
                data=True
            )
            if d["node_type"] == "file"
        ]

        chunk_group = [
            n for n, d in G.nodes(
                data=True
            )
            if d["node_type"] == "chunk"
        ]

        vector_group = [
            n for n, d in G.nodes(
                data=True
            )
            if d["node_type"] == "vector"
        ]

        retriever_group = [
            n for n, d in G.nodes(
                data=True
            )
            if d["node_type"] == "retriever"
        ]

        llm_group = [
            n for n, d in G.nodes(
                data=True
            )
            if d["node_type"] == "llm"
        ]

        # ----------------------------------------
        # Draw files
        # ----------------------------------------

        nx.draw_networkx_nodes(

            G,
            pos,

            nodelist=file_group,

            node_size=5000,

            node_color="lightblue",

            edgecolors="black",

            linewidths=1.5,

            ax=ax
        )

        # ----------------------------------------
        # Draw chunks
        # ----------------------------------------

        nx.draw_networkx_nodes(

            G,
            pos,

            nodelist=chunk_group,

            node_size=3000,

            node_color="lightgreen",

            edgecolors="black",

            ax=ax
        )

        # ----------------------------------------
        # Vector
        # ----------------------------------------

        nx.draw_networkx_nodes(

            G,
            pos,

            nodelist=vector_group,

            node_size=5000,

            node_color="plum",

            edgecolors="black",

            ax=ax
        )

        # ----------------------------------------
        # Retriever
        # ----------------------------------------

        nx.draw_networkx_nodes(

            G,
            pos,

            nodelist=retriever_group,

            node_size=5000,

            node_color="khaki",

            edgecolors="black",

            ax=ax
        )

        # ----------------------------------------
        # LLM
        # ----------------------------------------

        nx.draw_networkx_nodes(

            G,
            pos,

            nodelist=llm_group,

            node_size=5000,

            node_color="salmon",

            edgecolors="black",

            ax=ax
        )

        # ----------------------------------------
        # Edges
        # ----------------------------------------

        nx.draw_networkx_edges(

            G,

            pos,

            arrows=True,

            arrowsize=20,

            width=1.5,

            alpha=0.7,

            ax=ax
        )

        # ----------------------------------------
        # Labels
        # ----------------------------------------

        labels = {

            node:
            data["label"]

            for node, data

            in G.nodes(
                data=True
            )
        }

        nx.draw_networkx_labels(

            G,

            pos,

            labels=labels,

            font_size=8,

            font_weight="bold",

            ax=ax
        )

        ax.set_title(

            (
                f"Dynamic Document + RAG Graph\n"
                f"{len(source_files)} files | "
                f"{len(chunks)} chunks"
            ),

            fontsize=18,

            fontweight="bold"
        )

        ax.axis("off")

        plt.tight_layout()

        output_file = (
            "/content/dynamic_rag_graph.png"
        )

        plt.savefig(

            output_file,

            dpi=150,

            bbox_inches="tight"
        )

        plt.close(fig)

        return output_file

    except Exception as e:

        print(
            "Graph error:",
            traceback.format_exc()
        )

        return None


# ============================================================
# 9. LANGGRAPH STATE
# ============================================================

class RAGState(TypedDict):

    question: str

    context: str

    answer: str

    llm: Any


# ============================================================
# 10. LANGGRAPH RETRIEVAL NODE
# ============================================================

def retrieve_documents(
    state: RAGState
):

    question = state[
        "question"
    ]

    if VECTORSTORE is None:

        return {

            "context":
            "No documents have been uploaded."
        }

    docs = VECTORSTORE.similarity_search(

        question,

        k=4
    )

    context_parts = []

    for index, doc in enumerate(
        docs,
        start=1
    ):

        source = doc.metadata.get(

            "source_file",

            "Unknown"
        )

        context_parts.append(

            f"[Source {index}: {source}]\n"
            f"{doc.page_content}"
        )

    context = "\n\n".join(
        context_parts
    )

    return {

        "context":
        context
    }


# ============================================================
# 11. LANGGRAPH GENERATION NODE
# ============================================================

def generate_answer(
    state: RAGState
):

    question = state[
        "question"
    ]

    context = state[
        "context"
    ]

    llm = state[
        "llm"
    ]

    prompt = ChatPromptTemplate.from_messages(

        [

            (
                "system",

                """
You are a RAG document assistant.

Use ONLY the provided document context
to answer the question.

Rules:

- Do not invent facts.
- If the answer is not present,
  clearly say so.
- Be concise and accurate.
- Mention the source filename when helpful.

DOCUMENT CONTEXT:

{context}
"""
            ),

            (
                "human",

                "{question}"
            )
        ]
    )

    chain = prompt | llm

    response = chain.invoke(

        {

            "context":
            context,

            "question":
            question
        }
    )

    return {

        "answer":
        response.content
    }


# ============================================================
# 12. BUILD LANGGRAPH
# ============================================================

workflow = StateGraph(
    RAGState
)

workflow.add_node(
    "retrieve_documents",
    retrieve_documents
)

workflow.add_node(
    "generate_answer",
    generate_answer
)

workflow.add_edge(
    START,
    "retrieve_documents"
)

workflow.add_edge(
    "retrieve_documents",
    "generate_answer"
)

workflow.add_edge(
    "generate_answer",
    END
)

GRAPH = workflow.compile()


# ============================================================
# 13. LANGGRAPH WORKFLOW IMAGE
# ============================================================

def create_workflow_image():

    try:

        graph_data = GRAPH.get_graph()

        G = nx.DiGraph()

        for node in graph_data.nodes:

            name = str(node)

            if name == "__start__":

                label = "START"

            elif name == "__end__":

                label = "END"

            else:

                label = (
                    name
                    .replace(
                        "_",
                        "\n"
                    )
                    .title()
                )

            G.add_node(
                name,
                label=label
            )

        for edge in graph_data.edges:

            G.add_edge(

                str(edge.source),

                str(edge.target)
            )

        fig, ax = plt.subplots(

            figsize=(12, 5)
        )

        pos = nx.spring_layout(

            G,

            seed=42,

            k=2
        )

        nx.draw_networkx_nodes(

            G,

            pos,

            node_size=5000,

            node_color="lightblue",

            edgecolors="black",

            ax=ax
        )

        nx.draw_networkx_edges(

            G,

            pos,

            arrows=True,

            arrowsize=25,

            width=2,

            ax=ax
        )

        labels = {

            node: data["label"]

            for node, data in
            G.nodes(
                data=True
            )
        }

        nx.draw_networkx_labels(

            G,

            pos,

            labels=labels,

            font_size=11,

            font_weight="bold",

            ax=ax
        )

        ax.set_title(

            "LangGraph RAG Execution Workflow",

            fontsize=17,

            fontweight="bold"
        )

        ax.axis("off")

        plt.tight_layout()

        output_file = (
            "/content/langgraph_workflow.png"
        )

        plt.savefig(

            output_file,

            dpi=160,

            bbox_inches="tight"
        )

        plt.close(fig)

        return output_file

    except Exception as e:

        print(
            "Workflow graph error:",
            e
        )

        return None


WORKFLOW_IMAGE = (
    create_workflow_image()
)


# ============================================================
# 14. CREATE GROQ LLM
# ============================================================

def create_llm(
    groq_api_key
):

    if not groq_api_key:

        raise ValueError(
            "Please enter your Groq API key."
        )

    groq_api_key = (
        groq_api_key.strip()
    )

    if not groq_api_key:

        raise ValueError(
            "Please enter your Groq API key."
        )

    return ChatGroq(

        api_key=groq_api_key,

        model=
        "openai/gpt-oss-120b",

        temperature=0,

        max_tokens=2000,

        max_retries=2
    )


# ============================================================
# 15. CHAT FUNCTION
# ============================================================
#
# IMPORTANT:
#
# Your Gradio error:
#
# Data incompatible with messages format
#
# is fixed here by ALWAYS returning:
#
# [
#   {"role": "user", "content": "..."},
#   {"role": "assistant", "content": "..."}
# ]
#
# No tuple pairs are used.
# ============================================================

def chat_with_rag(

    message,

    history,

    groq_api_key
):

    try:

        history = history or []

        if not message or not message.strip():

            return history

        message = message.strip()

        # ----------------------------------------
        # Create LLM
        # ----------------------------------------

        llm = create_llm(
            groq_api_key
        )

        # ----------------------------------------
        # Knowledge base check
        # ----------------------------------------

        if VECTORSTORE is None:

            answer = (

                "Please upload your documents "
                "and click **Build Knowledge Base** "
                "before asking a question."
            )

        else:

            # ------------------------------------
            # Execute LangGraph
            # ------------------------------------

            result = GRAPH.invoke(

                {

                    "question":
                    message,

                    "context":
                    "",

                    "answer":
                    "",

                    "llm":
                    llm
                }
            )

            answer = result.get(

                "answer",

                "No answer generated."
            )

        # ----------------------------------------
        # IMPORTANT:
        # Messages format
        # ----------------------------------------

        history.append(

            {

                "role":
                "user",

                "content":
                message
            }
        )

        history.append(

            {

                "role":
                "assistant",

                "content":
                answer
            }
        )

        return history

    except Exception as e:

        print(
            traceback.format_exc()
        )

        history = history or []

        history.append(

            {

                "role":
                "user",

                "content":
                message
            }
        )

        history.append(

            {

                "role":
                "assistant",

                "content":
                f"❌ Error: {str(e)}"
            }
        )

        return history


# ============================================================
# 16. CLEAR CHAT
# ============================================================

def clear_chat():

    return []


# ============================================================
# 17. CSS
# ============================================================

custom_css = """

#app_title {
    text-align: center;
    font-size: 30px;
    font-weight: 700;
}

#app_subtitle {
    text-align: center;
    font-size: 15px;
    opacity: 0.8;
}

"""


# ============================================================
# 18. GRADIO UI
# ============================================================

with gr.Blocks(

    title=
    "LangChain + LangGraph Dynamic RAG"

) as demo:

    # ----------------------------------------
    # Header
    # ----------------------------------------

    gr.Markdown(

        """
# 🧠 LangChain + LangGraph Dynamic RAG

Upload documents → Build Knowledge Base →
Generate Dynamic Data Graph → Ask Questions
""",

        elem_id=
        "app_title"
    )

    gr.Markdown(

        """
**LLM:** `openai/gpt-oss-120b` via Groq
**Vector Store:** FAISS
**Embeddings:** all-MiniLM-L6-v2
""",

        elem_id=
        "app_subtitle"
    )


    # ========================================================
    # GROQ
    # ========================================================

    gr.Markdown(
        "## 🔐 Groq API Configuration"
    )

    with gr.Row():

        groq_api_key = gr.Textbox(

            label=
            "Groq API Key",

            placeholder=
            "Enter your Groq API key",

            type="password",

            scale=3
        )

        gr.Markdown(

            """
### Model

`openai/gpt-oss-120b`

The API key is supplied by the
user through the UI.
""",

            scale=1
        )


    # ========================================================
    # FILES
    # ========================================================

    gr.Markdown(
        "## 📚 Upload Documents"
    )

    with gr.Row():

        with gr.Column(
            scale=1
        ):

            files = gr.File(

                label=
                "PDF / DOCX / TXT / MD / CSV",

                file_count="multiple",

                file_types=[

                    ".pdf",

                    ".docx",

                    ".txt",

                    ".md",

                    ".csv"
                ],

                type="filepath"
            )

            build_button = gr.Button(

                "🚀 Build Knowledge Base",

                variant="primary"
            )


        with gr.Column(
            scale=1
        ):

            status = gr.Markdown(

                """
### 📂 Status

No documents loaded yet.
"""
            )


    # ========================================================
    # DYNAMIC DATA GRAPH
    # ========================================================

    gr.Markdown(
        "## 📊 Dynamic Document + RAG Graph"
    )

    gr.Markdown(

        """
This graph is regenerated from the actual uploaded data.

**File → Chunks → FAISS → Retriever → GPT-OSS-120B**
"""
    )

    document_graph = gr.Image(

        value=None,

        label=
        "Generated Data Graph",

        type="filepath"
    )


    # ========================================================
    # LANGGRAPH WORKFLOW
    # ========================================================

    gr.Markdown(
        "## 🔗 LangGraph Workflow"
    )

    workflow_graph = gr.Image(

        value=
        WORKFLOW_IMAGE,

        label=
        "LangGraph Nodes & Edges",

        type="filepath"
    )


    gr.Markdown(

        """
### LangGraph Execution

`START`
→
`retrieve_documents`
→
`generate_answer`
→
`END`
"""
    )


    # ========================================================
    # CHAT
    # ========================================================

    gr.Markdown(
        "## 💬 Chat with Your Documents"
    )

    chatbot = gr.Chatbot(

        label=
        "RAG Assistant",

        height=500,

        placeholder=
        "Ask a question about your uploaded documents..."
    )


    with gr.Row():

        message = gr.Textbox(

            label=
            "Your Question",

            placeholder=
            "Ask something about the uploaded documents...",

            scale=8
        )

        send_button = gr.Button(

            "Send",

            variant="primary",

            scale=1
        )

        clear_button = gr.Button(

            "Clear",

            scale=1
        )


    # ========================================================
    # EVENTS
    # ========================================================

    build_button.click(

        fn=
        build_knowledge_base,

        inputs=[
            files
        ],

        outputs=[

            status,

            document_graph
        ]
    )


    send_button.click(

        fn=
        chat_with_rag,

        inputs=[

            message,

            chatbot,

            groq_api_key
        ],

        outputs=[

            chatbot
        ]
    ).then(

        fn=
        lambda: "",

        outputs=[
            message
        ]
    )


    message.submit(

        fn=
        chat_with_rag,

        inputs=[

            message,

            chatbot,

            groq_api_key
        ],

        outputs=[

            chatbot
        ]
    ).then(

        fn=
        lambda: "",

        outputs=[
            message
        ]
    )


    clear_button.click(

        fn=
        clear_chat,

        inputs=[],

        outputs=[
            chatbot
        ]
    )


# ============================================================
# 19. LAUNCH
# ============================================================
#
# Gradio 6:
# CSS belongs in launch(), not Blocks().
# ============================================================

demo.launch(

    share=True,

    debug=True,

    css=custom_css
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/tmp/ipykernel_2793/911534552.py:48: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (


Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

... (7 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bf36632d0efcb0792b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
